####🏢 El Escenario Comercial: ¿Dónde estamos parados?
Superstore es una compañía líder en retail y comercio electrónico con operaciones a nivel nacional. La empresa comercializa tres grandes líneas de productos (Mobiliario, Suministros de Oficina y Tecnología) y atiende a tres perfiles de clientes claramente diferenciados: consumidores finales, oficinas corporativas y pequeñas empresas. En los últimos años, la compañía ha experimentado un crecimiento agresivo en su volumen de facturación, expandiendo su red logística a múltiples regiones y aplicando fuertes políticas promocionales para ganar cuota de mercado. Sin embargo, la alta dirección ha detectado una señal de alerta: el aumento en las ventas no se está traduciendo de forma lineal en un aumento de las ganancias netas.
####📊 El Encargo de la Gerencia
Ante esta situación, la Gerencia General de Superstore nos ha convocado como equipo de consultores analíticos para auditar la salud financiera y operativa de la organización. El mandato es claro: ir más allá de los reportes tradicionales de facturación y descubrir los factores ocultos que están afectando la rentabilidad.

In [37]:
# Importación de librerías requeridas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sympy as sp
from scipy.optimize import curve_fit

# Configuración opcional para mejorar la visualización de números flotantes
pd.set_option("display.float_format", lambda x: "%.2f" % x)

url_dataset = "https://raw.githubusercontent.com/Pitiki10/TP-GRUPAL---SuperStore/main/data/superstore.csv"
df = pd.read_csv(url_dataset, encoding="latin1")

print("I. ESTRUCTURA Y TIPOS DE DATOS DEL DATASET")
print("======================================================================")
df.info()

print("\n" + "-"*70 + "\n")

print("II. ESTADÍSTICA DESCRIPTIVA DE VARIABLES NUMÉRICAS")
print("======================================================================")
print(df.describe())

I. ESTRUCTURA Y TIPOS DE DATOS DEL DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ï»¿Row ID      9994 non-null   object 
 1   Order ID       7484 non-null   object 
 2   Order Date     7484 non-null   object 
 3   Ship Date      7484 non-null   object 
 4   Ship Mode      7484 non-null   object 
 5   Customer ID    7484 non-null   object 
 6   Customer Name  7484 non-null   object 
 7   Segment        7484 non-null   object 
 8   Country        7484 non-null   object 
 9   City           7484 non-null   object 
 10  State          7484 non-null   object 
 11  Postal Code    7484 non-null   float64
 12  Region         7484 non-null   object 
 13  Product ID     7484 non-null   object 
 14  Category       7484 non-null   object 
 15  Sub-Category   7484 non-null   object 
 16  Product Name   7484 non-null   object 
 17  Sales    

###**Limpieza y Transformación de Datos (Data Wrangling)**

In [39]:
# Renombrar las columnas con errores de caracteres o formato
df.rename(columns={
    "ï»¿Row ID": "Row ID",
    "Profit;": "Profit"
}, inplace=True)

# Mostrar estado inicial antes de eliminar nulos
print("I. CONTROL DE FILAS NULAS")
print("======================================================================")
print(f"Filas totales originales en el DataFrame: {len(df)}")

# Eliminar filas donde cualquiera de las columnas contenga un valor nulos
df.dropna(inplace=True)

print(f"Filas restantes después de eliminar registros nulos: {len(df)}")

print("\n" + "-"*70 + "\n")

# Evaluar si existen ID de fila repetidos (Duplicados)
print("II. EVALUACIÓN DE DUPLICADOS EN 'Row ID'")
print("======================================================================")
cantidad_duplicados = df["Row ID"].duplicated().sum()
print(f"Cantidad de IDs de fila repetidos detectados: {cantidad_duplicados}")

if cantidad_duplicados > 0:
    df.drop_duplicates(subset=["Row ID"], keep="first", inplace=True)
    print("▶️ Acción: Se eliminaron las filas duplicadas manteniendo la primera aparición.")
    print(f"Filas definitivas en el dataset: {len(df)}")
else:
    print("▶️ Acción: No se requieren eliminaciones. Cada fila posee un identificador único.")

# Transformacipon de la columna 'Profit' a tipo decimal (float)
df["Profit"] = df["Profit"].astype(str).str.rstrip(";")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")

print("\n" + "-"*70 + "\n")

print("III. VISTA GENERAL DEL DATASET LIMPIO")
print("======================================================================")
display(df.head())

I. CONTROL DE FILAS NULAS
Filas totales originales en el DataFrame: 7484
Filas restantes después de eliminar registros nulos: 7484

----------------------------------------------------------------------

II. EVALUACIÓN DE DUPLICADOS EN 'Row ID'
Cantidad de IDs de fila repetidos detectados: 0
▶️ Acción: No se requieren eliminaciones. Cada fila posee un identificador único.

----------------------------------------------------------------------

III. VISTA GENERAL DEL DATASET LIMPIO


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.00,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.00,0.00,41.9136;
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.00,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.00,0.00,6.8714;
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.00,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5.00,0.45,-383.031;
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.00,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2.00,0.20,2.5164;
6,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.00,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4.00,0.00,1.9656;


In [29]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.00,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.00,0.00,41.9136;
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.00,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.00,0.00,6.8714;
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.00,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5.00,0.45,-383.031;
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.00,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2.00,0.20,2.5164;
6,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.00,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4.00,0.00,1.9656;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9987,9988,CA-2017-163629,11/17/2017,11/21/2017,Standard Class,RA-19885,Ruben Ausman,Corporate,United States,Athens,Georgia,30605.00,South,TEC-AC-10001539,Technology,Accessories,Logitech G430 Surround Sound Gaming Headset wi...,79.99,1.00,0.00,28.7964;
9988,9989,CA-2017-163629,11/17/2017,11/21/2017,Standard Class,RA-19885,Ruben Ausman,Corporate,United States,Athens,Georgia,30605.00,South,TEC-PH-10004006,Technology,Phones,Panasonic KX - TS880B Telephone,206.10,5.00,0.00,55.647;
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,Florida,33180.00,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.25,3.00,0.20,4.1028;
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627.00,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.96,2.00,0.00,15.6332;


### **🎯 Interrogante Principal de Alto Impacto**
###**Pregunta**: ¿De qué manera la política de descuentos altera las funciones de ingreso y beneficio de Superstore, y cuál es el nivel óptimo de descuento que maximiza la ganancia neta antes de que la elasticidad de la demanda convierta las operaciones en pérdidas?

###**Fundamentación Económica e Hipótesis**: Proponemos que la Ganancia Total (TB) no tiene un comportamiento lineal respecto al descuento, sino cóncavo (forma de U invertida). Estimamos que existe un umbral crítico de descuento donde la elasticidad-precio de la demanda deja de ser favorable; a partir de ese punto, la tasa de cambio del ingreso es menor a la tasa de cambio del costo operativo, provocando que la derivada del beneficio se vuelva negativa y destruya el margen unitario de la compañía. Al modelar analíticamente el Ingreso Total (TR) y el Costo Total (TC) en función del descuento, mediante derivadas y condiciones de primer y segundo orden, demostraremos que existe un punto de quiebre financiero

####🔄 **Preguntas Complementarias (Análisis de Mercado y Logística)**
Para robustecer el diagnóstico, aislaremos el problema principal cruzando variables de catálogo, geográficas y operativas bajo las siguientes ópticas:
1. Margen, Rentabilidad y Participación por Catálogo: ¿Cómo se distribuye la ganancia total y la participación porcentual del beneficio entre las distintas categorías, y qué subcategorías actúan como "anclas" financieras?. Se calculará el margen unitario y el aporte de cada sector al beneficio global para identificar si Tecnología subsidia las pérdidas ocultas de Mobiliario.

2. Análisis de Mercado ante Shocks Externos y Logística Regional: ¿Cómo reacciona la demanda de las categorías críticas en las regiones periféricas ante un shock externo en los costos logísticos, y cómo afecta esto a las tasas de cambio de los envíos?